<a href="https://colab.research.google.com/github/mf2056/Dissertation/blob/main/Baseline_ResUNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
# Load the data
X_train = np.load('/content/drive/MyDrive/X_train.npy')
Y_train_mask = np.load('/content/drive/MyDrive/Y_train_mask.npy')
X_test = np.load('/content/drive/MyDrive/X_test.npy')
Y_test_mask = np.load('/content/drive/MyDrive/Y_test_mask.npy')

In [ ]:
import numpy as np
import tensorflow as tf

# Define the ESA class map
class_map = {
    10: 0, #(Tree cover, "#006400")
    20: 1, #(Shrubland, "#ffbb22")
    30: 2, #(Grassland, "#ffff4c")
    40: 3, #(Cropland, "#f096ff")
    60: 3, #(Bare / Sparse vegetation, "#b4b4b4")  Combined with Cropland
    50: 4, #(Built-up, "#fa0000")
    80: 5, #(Permanent water bodies, "#0064ff")
    90: 5, #(Herbaceous wetland, "#0096a0")   Combined with Water Bodies
}

# map the high ESA numbers to 0, 1, 2, 3, 4, 5 across all pixels
lut = np.full(91, -1, dtype=np.int32)
for old_id, new_id in class_map.items():
    lut[old_id] = new_id

# Apply mapping to the whole 3D array (Spatial Mapping)
Y_train_ready = lut[Y_train_mask]
Y_test_ready = lut[Y_test_mask]

print(f"Unique IDs in merged train mask: {np.unique(Y_train_ready)}")

Unique IDs in merged train mask: [0 1 2 3 4 5]


In [ ]:
num_classes = 6
Y_train_cat = tf.keras.utils.to_categorical(Y_train_ready, num_classes=num_classes)
Y_test_cat = tf.keras.utils.to_categorical(Y_test_ready, num_classes=num_classes)

print(f"X_train shape: {X_train.shape}") # no.of bands(7)
print(f"Y_train shape: {Y_train_cat.shape}") # no.of classes(6)

X_train shape: (639, 256, 256, 7)
Y_train shape: (639, 256, 256, 6)


In [ ]:
# Converting for RAM efficiency

X_train = X_train.astype('float32')
Y_train_cat = Y_train_cat.astype('float32')
X_test = X_test.astype('float32')
Y_test_cat = Y_test_cat.astype('float32')

In [ ]:
import tensorflow.keras.backend as K

def residual_block(x, filters, dropout_rate=0.3):
    shortcut = x

    # First Convolution
    x = layers.Conv2D(filters, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Second Convolution
    x = layers.Conv2D(filters, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)

    # Add Dropout BEFORE the addition to prevent memorization
    x = layers.Dropout(dropout_rate)(x)

    # Adjust shortcut dimensions if necessary
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, (1, 1), padding='same')(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    # The Addition
    x = layers.add([x, shortcut])
    x = layers.Activation('relu')(x)
    return x

def build_res_unet(input_shape=(256, 256, 7), num_classes=6):
    inputs = layers.Input(input_shape)

    # Encoder
    s1 = residual_block(inputs, 32)
    p1 = layers.MaxPooling2D((2, 2))(s1)

    s2 = residual_block(p1, 64)
    p2 = layers.MaxPooling2D((2, 2))(s2)

    s3 = residual_block(p2, 128)
    p3 = layers.MaxPooling2D((2, 2))(s3)

    # Bridge
    b1 = residual_block(p3, 256)

    # Decoder
    u1 = layers.Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(b1)
    u1 = layers.concatenate([u1, s3])
    d1 = residual_block(u1, 128)

    u2 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(d1)
    u2 = layers.concatenate([u2, s2])
    d2 = residual_block(u2, 64)

    u3 = layers.Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(d2)
    u3 = layers.concatenate([u3, s1])
    d3 = residual_block(u3, 32)

    outputs = layers.Conv2D(num_classes, (1, 1), activation='softmax')(d3)
    return models.Model(inputs, outputs)

model = build_res_unet()
print("ResU-Net Built successfully!")

ResU-Net Built successfully!


In [ ]:
import tensorflow.keras.backend as K
import tensorflow as tf

def weighted_categorical_crossentropy(weights, label_smoothing=0.1):
    weights = K.variable(weights)

    def loss(y_true, y_pred):
        # Apply Label Smoothing manually
        num_classes = K.cast(K.shape(y_true)[-1], y_true.dtype)
        smooth_y_true = y_true * (1.0 - label_smoothing) + (label_smoothing / num_classes)

        # Standard Categorical Crossentropy math
        y_pred /= K.sum(y_pred, axis=-1, keepdims=True)
        y_pred = K.clip(y_pred, K.epsilon(), 1.0 - K.epsilon())

        # Apply the Weights to the smoothed labels
        weighted_loss = smooth_y_true * K.log(y_pred) * weights
        return -K.sum(weighted_loss, axis=-1)

    return loss

In [ ]:
from sklearn.utils import class_weight
from sklearn.metrics import f1_score
import numpy as np

# flatten the masks so compute_class_weight can see every single pixel
flat_y = Y_train_ready.flatten()
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(flat_y),
    y=flat_y
)


In [ ]:
print("Class Weights:", weights)

Class Weights: [1.11912802 1.041065   0.75546067 0.97782913 1.32307165 0.95812405]


In [ ]:
# Custom Table Logger
class TableLogger(tf.keras.callbacks.Callback):
    def __init__(self, val_data):
        super().__init__()
        self.X_val, self.y_val_cat = val_data

    def on_train_begin(self, logs=None):
        print(f"\n{'Epoch':<6} | {'Train Loss':<10} | {'Val Loss':<10} | {'Acc':<8} | {'F1 (Macro)':<10}")
        print("-" * 60)

    def on_epoch_end(self, epoch, logs=None):
        val_logits = self.model.predict(self.X_val, verbose=0, batch_size=16)


        val_preds = np.argmax(val_logits, axis=-1).flatten()
        val_true = np.argmax(self.y_val_cat, axis=-1).flatten()

        # Calculate F1
        val_f1 = f1_score(val_true, val_preds, average='macro')

        # Get values from logs
        train_loss = logs.get('loss', 0)
        val_loss = logs.get('val_loss', 0)
        val_acc = logs.get('val_accuracy', 0)

        print(f"{epoch+1:<6} | {train_loss:<10.4f} | {val_loss:<10.4f} | {val_acc:<8.4f} | {val_f1:<10.4f}")


In [ ]:
# Re-build and Compile
model = build_res_unet(input_shape=(256, 256, 7), num_classes=6)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=weighted_categorical_crossentropy(weights, label_smoothing=0.1),
    metrics=['accuracy']
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=3,
    min_lr=1e-7
)

table_logger = TableLogger(val_data=(X_test, Y_test_cat))

# Start training
history = model.fit(
    X_train, Y_train_cat,
    validation_data=(X_test, Y_test_cat),
    epochs=20,
    batch_size=16,
    callbacks=[table_logger, reduce_lr],
    verbose=0
)


Epoch  | Train Loss | Val Loss   | Acc      | F1 (Macro)
------------------------------------------------------------
1      | 2.2378     | 1.7964     | 0.2000   | 0.1291    
2      | 2.0286     | 1.7988     | 0.2302   | 0.1900    
3      | 1.8614     | 1.8088     | 0.2052   | 0.1701    
4      | 1.7136     | 1.8175     | 0.1895   | 0.1398    
5      | 1.6299     | 1.8225     | 0.1843   | 0.1277    
6      | 1.5944     | 1.8193     | 0.1877   | 0.1298    
7      | 1.5731     | 1.8095     | 0.1971   | 0.1386    
8      | 1.5544     | 1.7890     | 0.2093   | 0.1496    
9      | 1.5589     | 1.7553     | 0.2231   | 0.1626    
10     | 1.5498     | 1.7116     | 0.2446   | 0.1872    
11     | 1.5481     | 1.6619     | 0.2895   | 0.2438    
12     | 1.5413     | 1.6131     | 0.3848   | 0.3406    
13     | 1.5374     | 1.5696     | 0.4399   | 0.3946    
14     | 1.5361     | 1.5293     | 0.4816   | 0.4390    
15     | 1.5576     | 1.4863     | 0.5142   | 0.4766    
16     | 1.5324     | 1.44

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, accuracy_score, f1_score
import gc

def pixels_to_patch_labels(y_array):
    """ Converts (N, 256, 256) pixel masks into (N,) majority-class labels. """
    return np.array([np.argmax(np.bincount(patch.flatten(), minlength=6)) for patch in y_array])

print("Predicting U-Net Masks...")
test_preds_raw = model.predict(X_test, batch_size=8, verbose=1)
test_preds_pixels = np.argmax(test_preds_raw, axis=-1)

train_preds_raw = model.predict(X_train, batch_size=8, verbose=1)
train_preds_pixels = np.argmax(train_preds_raw, axis=-1)

# Free up RAM immediately
del test_preds_raw, train_preds_raw
gc.collect()

y_train_patch_true = pixels_to_patch_labels(Y_train_ready)
y_train_patch_pred = pixels_to_patch_labels(train_preds_pixels)
y_test_patch_true = pixels_to_patch_labels(Y_test_ready)
y_test_patch_pred = pixels_to_patch_labels(test_preds_pixels)

# Calculate Metrics
train_acc = accuracy_score(y_train_patch_true, y_train_patch_pred)
test_acc = accuracy_score(y_test_patch_true, y_test_patch_pred)
test_f1_macro = f1_score(y_test_patch_true, y_test_patch_pred, average="macro")


merged_names = ["Tree cover", "Shrubland", "Grassland", "Cropland", "Built-up", "Water"]

print(f"Train Accuracy (Patch): {train_acc:.4f}")
print(f"Test Accuracy (Patch):  {test_acc:.4f}")
print(f"Test Macro F1 (Patch):  {test_f1_macro:.4f}")

print("\n CLASSIFICATION REPORT ")
print(classification_report(y_test_patch_true, y_test_patch_pred, target_names=merged_names))

Predicting U-Net Masks...
16/16 ━━━━━━━━━━━━━━━━━━━━ 13s 444ms/step
80/80 ━━━━━━━━━━━━━━━━━━━━ 11s 143ms/step
Train Accuracy (Patch): 0.6651
Test Accuracy (Patch):  0.7200
Test Macro F1 (Patch):  0.6982

 CLASSIFICATION REPORT 
              precision    recall  f1-score   support

  Tree cover       1.00      0.67      0.80         6
   Shrubland       0.41      1.00      0.58        13
   Grassland       0.50      0.14      0.22        28
    Cropland       0.70      0.97      0.81        29
    Built-up       1.00      0.64      0.78        22
       Water       1.00      1.00      1.00        27

    accuracy                           0.72       125
   macro avg       0.77      0.74      0.70       125
weighted avg       0.76      0.72      0.69       125

